In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, current_timestamp
import requests
import pandas as pd
import io

spark.sql("USE CATALOG scottish_housing_ws")
spark.sql("USE SCHEMA bronze")

In [0]:
url = "https://www.ons.gov.uk/generator?format=csv&uri=/employmentandlabourmarket/peopleinwork/earningsandworkinghours/timeseries/kab9/emp"

headers = {"User-Agent": "Mozilla/5.0"}
response = requests.get(url, headers=headers)
response.raise_for_status()

df_pd = pd.read_csv(io.StringIO(response.text))
print(df_pd.shape)
print(df_pd.head(10))
print(df_pd.columns.tolist())

In [0]:
df_pd_clean = pd.read_csv(
    io.StringIO(response.text),
    skiprows=8,
    header=None,
    names=["time_period", "avg_weekly_earnings_gbp"]
)

print(df_pd_clean.shape)
print(df_pd_clean.head(10))
print(df_pd_clean.tail(5))


In [0]:
df_bronze = spark.createDataFrame(df_pd_clean)

(
    df_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("bronze.ons_awe")
)

In [0]:
spark.sql("SELECT * FROM bronze.ons_awe LIMIT 5").show()
spark.sql("SELECT COUNT(*) FROM bronze.ons_awe").show()

In [0]:
# bronze.ons_awe loaded
# Source: ONS AWE time series KAB9
# 444 rows, mixed granularity (annual, quarterly, monthly)
# Metadata rows skipped at read time
# Filtering to monthly rows happens in silver